In [28]:
import os
from dotenv import load_dotenv

import numpy as np
from openai import OpenAI
import pandas as pd
from tqdm.notebook import tqdm

load_dotenv()

True

In [11]:
client = OpenAI(
    base_url=os.getenv("GENERATION_API_URL"),
    api_key=os.getenv("SCW_SECRET_KEY"),
)

In [12]:
clusters = pd.read_csv('cluster_representatives_2026-03-10.csv')

In [16]:
clusters.text.str.len().max()

np.int64(532)

In [22]:
batch_size = 500
results = []
for i in tqdm(range(0, len(clusters), batch_size)):
    batch = clusters.iloc[i:i+batch_size]
    response = client.embeddings.create(
        input=batch['text'].tolist(),
        model="qwen3-embedding-8b",
    )
    results.extend(response.data)

  0%|          | 0/6 [00:00<?, ?it/s]

In [30]:
embeddings = np.array([r.embedding for r in results])

In [31]:
embeddings.shape

(2625, 4096)

In [32]:
clusters['embedding'] = list(embeddings)

In [34]:
clusters.to_parquet('clusters_representatives_with_embeddings_Qwen3-8B.parquet', index=False)